In [ ]:
import pandas as pd
import os, sys

class Classify():

    print_columns_d = ['index', 'Object', 'Sanitised', 'Reduced', 'RA', 'DEC', 
                    'SNR', 'MJD-OBS', 'EXPTIME', 'New Groups', 'Min_SNR', 'RV_pos' ]

    print_columns_o = ['index', 'Object', 'Sanitised', 'Reduced', 'RA', 'DEC', 'SNR', 'MJD-OBS',
     'EXPTIME', 'New Groups', 'Min_SNR', 'RV_pos', 'Abs_width', 'Abs_depth' ]

    def __init__(self, parameters, res_path):
        
        self.param = [parameters["threshold"], parameters["med_filter_bin"],
                        parameters["rv_min"], parameters["rv_max"], parameters["cutoff"],
                         parameters["width_filt"]]

        self.res_path = res_path
        self.cand_report_path = res_path + 'candidate_report.pkl'

        status = ['flagged', 'not_candidate_but_real', 'not_candidate_but_junk', 'candidate']

        for st in status:
            if os.path.exists(res_path + '{}/'.format(st)) == False:
                os.mkdir(res_path + '{}/'.format(st))

        if os.path.exists(self.cand_report_path):
            self.candidate_report = pd.read_pickle(self.cand_report_path)
        else:
            self.candidate_report = pd.DataFrame(data={'Target':[], 'Status':[], 'Parameters':[]}).astype(object)

        self.target_name = None
        self.target_info = None
        self.detection_info = None
        self.previous_report = None
        self.current_status = None

        self.skipped = []
        self.flagged = False

    def candidate_info(self, target):
        ''' Load all information from a specific target.
            Return -> Has the target already been looked at? True/False '''
        
        self.current_status = None
        self.target_name = target

        self.previous_report = self.candidate_report[self.candidate_report['Target'] == target]

        if len(self.previous_report) == 1:
            classified = True
            if 'flagged' in self.previous_report.Status.to_numpy():
                self.flagged = True
            else:
                self.flagged = False
        elif len(self.previous_report) > 1:
            raise ValueError('Candidate:{} has more than one entry in the DataFrame.'.format(target))
        else:
            classified = False
            self.flagged = False

        return classified

    def classify(self):
        ''' Classify target according to its status - ie. candidate/not candidate/flagged'''

        if self.current_status == 'skipped':
            self.skipped.append(self.target_name)
            return None

        elif self.flagged is not True:
            cand_report = self.candidate_report
            cand_report.loc[len(cand_report.index)] = [self.target_name, self.current_status, self.param]
            self.candidate_report = cand_report.astype(object)

        else:
            self.candidate_report.iloc[self.previous_report.index.values[0]]['Status'] = self.current_status
        
        return self.res_path + '{}/'.format(self.current_status)

    def ask_user(self):
        ''' Prompt the user in the notebook cell output for a classification.
            Keys: y=candidate, n=not candidate (real), j=junk, w=flag, space/s=skip,
                  enter/q=quick save, esc/x=quit+save, d=print df, o=print detections '''
        print('\nClassify {} — y=candidate | n=not cand (real) | j=junk | w=flag | s=skip | q=save | x=quit+save | d=print df | o=print detections'.format(self.target_name))
        while True:
            key = input('>>> ').strip().lower()

            if key == 'y':
                self.current_status = 'candidate'
                print('{}: CANDIDATE'.format(self.target_name))
                break

            elif key == 'n':
                self.current_status = 'not_candidate_but_real'
                print('{}: NOT A CANDIDATE — real astrophysical variability'.format(self.target_name))
                break

            elif key == 'j':
                self.current_status = 'not_candidate_but_junk'
                print('{}: NOT A CANDIDATE — junk'.format(self.target_name))
                break

            elif key == 'w':
                self.current_status = 'flagged'
                print('{}: FLAGGED'.format(self.target_name))
                break

            elif key in ('s', ''):
                self.current_status = 'skipped'
                print('{}: SKIPPED'.format(self.target_name))
                break

            elif key == 'q':
                print('Saving progress...')
                self.candidate_report.to_pickle(self.cand_report_path)
                self.candidate_report.to_html(self.res_path + 'Report.html')
                print('Saved!')

            elif key == 'x':
                self.candidate_report.to_pickle(self.cand_report_path)
                self.candidate_report.to_html(self.res_path + 'Report.html')
                print('Saved! Quitting.')
                sys.exit()

            elif key == 'd':
                avail = [c for c in self.print_columns_d if c in self.target_info.columns]
                print(self.target_info[avail])

            elif key == 'o':
                if self.detection_info is not None:
                    avail = [c for c in self.print_columns_o if c in self.detection_info.columns]
                    print(self.detection_info[avail])
                else:
                    print('No detection info available.')

            else:
                print('Unknown key "{}". Try again.'.format(key))

    def onkey(self, event):
        ''' Legacy matplotlib key-press handler (not used in notebook mode). '''
        key_map = {
            'y': ('candidate', '{}: Currently CANDIDATE'),
            'n': ('not_candidate_but_real', '{}: Currently NOT A CANDIDATE: Real astrophysical variability'),
            'j': ('not_candidate_but_junk', '{}: Currently NOT A CANDIDATE: Junk'),
            'w': ('flagged', '{}: Currently FLAGGED'),
            ' ': ('skipped', '{}: Currently SKIPPED'),
        }
        if event.key in key_map:
            self.current_status, msg = key_map[event.key]
            print(msg.format(self.target_name))
        elif event.key == 'enter':
            print('Saving progress...')
            self.candidate_report.to_pickle(self.cand_report_path)
            self.candidate_report.to_html(self.res_path + 'Report.html')
            print('Saved!')
        elif event.key == 'escape':
            self.candidate_report.to_pickle(self.cand_report_path)
            self.candidate_report.to_html(self.res_path + 'Report.html')
            print('Saved!')
            sys.exit()
        elif event.key == 'd':
            avail = [c for c in self.print_columns_d if c in self.target_info.columns]
            print(self.target_info[avail])
        elif event.key == 'o':
            if self.detection_info is not None:
                avail = [c for c in self.print_columns_o if c in self.detection_info.columns]
                print(self.detection_info[avail])
